In [1]:
import re, time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import unicodedata

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score
from functools import lru_cache
from wordfreq import zipf_frequency

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.inspection import permutation_importance

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

SEED = 42

# RANK_AVG chooses how the five models are combined:
# True  = rank-average (each model's scores become 0-1 positions, then averaged)
# False = probability average (each model's P(AI) averaged directly)
RANK_AVG = False
DATA_DIR = Path("data")
SUB_DIR = Path("submissions")
SUB_DIR.mkdir(exist_ok=True)


In [2]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

tr_df, val_df = train_test_split(train_df, test_size=2000,stratify=train_df["label"], random_state=SEED)

X_tr,  y_tr  = tr_df["text"].tolist(), tr_df["label"].to_numpy()
X_val, y_val = val_df["text"].tolist(), val_df["label"].to_numpy()
X_all, y_all = train_df["text"].tolist(), train_df["label"].to_numpy()

X_test = test_df["text"].tolist()

# 5-fold splitter, used for tuning settings inside the 8,000 training rows
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

## Stylometric features
67 features are calculated from each passage describing how it is written, for example:
sentence-length variation ("burstiness"), word length and readability, vocabulary richness,
how common the words are (wordfreq), punctuation rates, and function-word rates. Whitespace is collapsed to single spaces first and blank lines are removed.


In [3]:
# Split sentences based on end punctuation.
SENT_SPLIT = re.compile(r"(?<=[.!?])[\"')\]]*\s+|\n+")
# A word is letters, optionally with an apostrophe for contraction.
WORD_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")

# Grammar words that carry little meaning, including pronouns
FUNCTION_WORDS = set("""
a about above after again against all am an and any are as at be because been before being below between both but by
can could did do does doing down during each few for from further had has have having he her here hers herself him
himself his how i if in into is it its itself just me more most my myself no nor not now of off on once only or other
our ours ourselves out over own same she should so some such than that the their theirs them themselves then there
these they this those through to too under until up very was we were what when where which while who whom why will
with would you your yours yourself yourselves may might must shall upon whether although though yet
""".split())
FIRST_PERSON = {"i", "me", "my", "mine", "myself", "we", "us", "our", "ours", "ourselves"}
SECOND_PERSON = {"you", "your", "yours", "yourself", "yourselves"}
HEDGES = {"may", "might", "could", "perhaps", "possibly", "likely", "potentially", "generally", "typically", "often"}
# Some AI transition-heavy words.
AI_CONNECTIVES = {"additionally", "furthermore", "moreover", "overall", "however", "ultimately", "notably",
                  "consequently", "therefore", "thus", "importantly", "significantly", "various", "crucial",
                  "essential", "ensure", "enhance", "comprehensive", "valuable", "unique"}

def collapse_text(t):
    return re.sub(r"\s+", " ", t).strip()

In [4]:
# How common a word is in English.
@lru_cache(maxsize=None)
def zipf(w):
    return zipf_frequency(w, "en")

# Rough syllable count
def syllables(w):
    w = w.lower()
    n = len(re.findall(r"[aeiouy]+", w))
    if w.endswith("e") and n > 1 and not w.endswith(("le", "ee")):
        n -= 1
    return max(n, 1)

def safe_cv(x):
    x = np.asarray(x, float)
    if len(x) > 1 and x.mean() > 0:
        return float(x.std() / x.mean())
    return 0.0

def mattr(tokens, w=50):
    if len(tokens) <= w:
        return len(set(tokens)) / max(len(tokens), 1)
    c = Counter(tokens[:w])
    vals = [len(c)]
    for i in range(w, len(tokens)):
        c[tokens[i]] += 1
        old = tokens[i - w]
        c[old] -= 1
        if c[old] == 0:
            del c[old]
        vals.append(len(c))
    return float(np.mean(vals)) / w

def lexical_burstiness(tokens, min_count=3):
    positions = {}
    for i, t in enumerate(tokens):
        positions.setdefault(t, []).append(i)
    scores = []
    for p in positions.values():
        if len(p) >= min_count:
            gaps = np.diff(p).astype(float)
            m, s = gaps.mean(), gaps.std()
            if m + s > 0:
                scores.append((s - m) / (s + m))
    return float(np.mean(scores)) if scores else 0.0

In [5]:
def stylometric_features(text):
    f = {}
    n_chars = max(len(text), 1)
    sents = [s.strip() for s in SENT_SPLIT.split(text) if s and WORD_RE.search(s)]
    words = WORD_RE.findall(text)
    lw = [w.lower() for w in words]
    nw = max(len(words), 1)
    per100 = 100.0 / nw

    f["n_words"] = len(words)

    # Sentence rhythm ("burstiness"): humans vary sentence length more
    sl = np.array([len(WORD_RE.findall(s)) for s in sents] or [0], float)
    f["sent_len_mean"] = sl.mean()
    f["sent_len_std"] = sl.std()
    f["sent_len_cv"] = safe_cv(sl)
    f["sent_len_max"] = sl.max()
    f["sent_len_min"] = sl.min()
    f["frac_short_sents"] = float((sl < 8).mean())
    f["frac_long_sents"] = float((sl > 30).mean())
    f["sent_len_diff_mean"] = float(np.abs(np.diff(sl)).mean()) if len(sl) > 1 else 0.0

    # Word length and readability
    wl = np.array([len(w) for w in words] or [0], float)
    f["word_len_mean"] = wl.mean()
    f["word_len_std"] = wl.std()
    f["frac_long_words"] = float((wl >= 7).mean())
    syl = np.array([syllables(w) for w in words] or [1], float)
    f["syll_per_word"] = syl.mean()
    f["flesch"] = 206.835 - 1.015 * sl.mean() - 84.6 * syl.mean()
    f["fk_grade"] = 0.39 * sl.mean() + 11.8 * syl.mean() - 15.59

    # Vocabulary richness and repetition
    cnt = Counter(lw)
    f["ttr_first100"] = len(set(lw[:100])) / max(min(100, len(lw)), 1)
    f["mattr50"] = mattr(lw)
    f["hapax_ratio"] = sum(1 for c in cnt.values() if c == 1) / max(len(cnt), 1)
    s2 = sum(c * c for c in cnt.values())
    f["yule_k"] = 1e4 * (s2 - nw) / (nw * nw)
    f["lex_burstiness"] = lexical_burstiness(lw)
    bigrams = list(zip(lw, lw[1:]))
    bc = Counter(bigrams)
    f["repeated_bigram_frac"] = sum(c for c in bc.values() if c > 1) / max(len(bigrams), 1)
    first_words = [WORD_RE.findall(s)[0].lower() for s in sents]
    f["sent_start_diversity"] = len(set(first_words)) / max(len(first_words), 1)

    # Word predictability from wordfreq (Zipf scale: 0 = unseen word, about 7.7 = "the")
    z = np.array([zipf(w) for w in lw] or [0], float)
    f["zipf_mean"] = z.mean()
    f["zipf_std"] = z.std()
    f["frac_rare_words"] = float((z < 3).mean())
    f["frac_oov_words"] = float((z == 0).mean())
    content = np.array([zipf(w) for w in lw if w not in FUNCTION_WORDS] or [0], float)
    f["zipf_content_mean"] = content.mean()
    # How much predictability changes from sentence to sentence
    sent_z = [np.mean([zipf(w.lower()) for w in WORD_RE.findall(s)]) for s in sents]
    f["sent_zipf_std"] = float(np.std(sent_z)) if len(sent_z) > 1 else 0.0

    # Punctuation, per 100 words
    for name, ch in [("comma", ","), ("semicolon", ";"), ("colon", ":"), ("em_dash", "—"), ("en_dash", "–"),
                     ("paren", "("), ("exclam", "!"), ("question", "?"), ("period", ".")]:
        f[f"p_{name}"] = text.count(ch) * per100
    f["p_quote"] = (text.count('"') + text.count("“") + text.count("”")) * per100
    f["p_apostrophe"] = (text.count("'") + text.count("’")) * per100
    f["p_ellipsis"] = (text.count("...") + text.count("…")) * per100
    f["p_hyphen_word"] = len(re.findall(r"[A-Za-z]-[A-Za-z]", text)) * per100
    f["punct_per_sent"] = len(re.findall(r"[^\w\s]", text)) / max(len(sents), 1)

    # Character-level habits
    f["upper_frac"] = sum(c.isupper() for c in text) / n_chars
    f["digit_frac"] = sum(c.isdigit() for c in text) / n_chars
    f["non_ascii_frac"] = sum(ord(c) > 127 for c in text) / n_chars
    nfkc = unicodedata.normalize("NFKC", text)
    f["nfkc_changed_frac"] = sum(a != b for a, b in zip(text, nfkc)) / n_chars
    f["caps_word_frac"] = sum(1 for w in words if len(w) > 1 and w.isupper()) / nw
    f["titlecase_midsent_frac"] = sum(1 for a, b in zip(words, words[1:]) if b[0].isupper()) / nw

    # Function words and discourse markers, per 100 words
    f["function_word_rate"] = sum(w in FUNCTION_WORDS for w in lw) * per100
    f["first_person_rate"] = sum(w in FIRST_PERSON for w in lw) * per100
    f["second_person_rate"] = sum(w in SECOND_PERSON for w in lw) * per100
    f["contraction_rate"] = sum("'" in w for w in lw) * per100
    f["hedge_rate"] = sum(w in HEDGES for w in lw) * per100
    f["ai_connective_rate"] = sum(w in AI_CONNECTIVES for w in lw) * per100
    for w in ["the", "of", "and", "a", "to", "in", "is", "that", "it", "this", "with", "as", "but"]:
        f[f"fw_{w}"] = cnt.get(w, 0) * per100
    return f

In [6]:
def stylo_frame(texts):
    return pd.DataFrame([stylometric_features(collapse_text(t)) for t in texts])

t0 = time.time()
S_tr = stylo_frame(X_tr)
S_val = stylo_frame(X_val)
print(f"{S_tr.shape[1]} stylometric features | train {S_tr.shape} | val {S_val.shape} | {time.time() - t0:.0f}s")
S_tr.describe().T[["mean", "std", "min", "max"]].round(2).head(15)


67 stylometric features | train (8000, 67) | val (2000, 67) | 10s


,mean,std,min,max
n_words,204.79,60.01,2.00,390.00
sent_len_mean,26.56,29.25,1.00,339.00
sent_len_std,10.96,11.25,0.00,153.50
sent_len_cv,0.48,0.26,0.00,1.88
sent_len_max,48.32,36.43,1.00,339.00
sent_len_min,13.44,29.09,1.00,339.00
frac_short_sents,0.10,0.14,0.00,1.00
frac_long_sents,0.20,0.24,0.00,1.00
sent_len_diff_mean,12.94,17.76,0.00,307.00
word_len_mean,4.88,0.66,2.24,13.32


## Model 5: stylometric gradient boosting (HGB)
4 Settings with 5-fold cross validation in order to determine the best hyperparameters.


In [7]:
results = []
val_scores = {}

def record(name, cv_auc, val_score, params, seconds):
    val_scores[name] = val_score
    val_auc = roc_auc_score(y_val, val_score)
    results.append({"model": name, "cv_auc": round(cv_auc, 4), "val_auc": round(val_auc, 4),
                    "params": params, "seconds": round(seconds)})
    print(f"{name}: CV AUROC {cv_auc:.4f}, val {val_auc:.4f}, {params} | {seconds:.0f}s")

# learning_rate: how much each new tree corrects the previous ones (smaller = more cautious)
# max_leaf_nodes: how many end points ("leaves") each tree can have (more = more complex rules)
HGB_GRID = {"learning_rate": [0.03, 0.1], "max_leaf_nodes": [15, 31]}

t0 = time.time()
gs = GridSearchCV(HistGradientBoostingClassifier(max_iter=400, random_state=SEED),
                  HGB_GRID, scoring="roc_auc", cv=cv, n_jobs=-1)
gs.fit(S_tr, y_tr)

stylo_hgb = gs.best_estimator_
stylo_hgb_params = gs.best_params_

record("stylo_hgb", gs.best_score_, stylo_hgb.predict_proba(S_val)[:, 1], stylo_hgb_params, time.time() - t0)


stylo_hgb: CV AUROC 0.9606, val 0.9701, {'learning_rate': 0.1, 'max_leaf_nodes': 31} | 11s


In [8]:
# Permutation importance: AUROC drop when each feature is shuffled
perm = permutation_importance(stylo_hgb, S_val, y_val, scoring="roc_auc",
                              n_repeats=5, random_state=SEED, n_jobs=-1)

imp = pd.DataFrame({
    "importance": perm.importances_mean,
    "direction": ["AI higher" if roc_auc_score(y_val, S_val[c]) > 0.5 else "AI lower" for c in S_val.columns],
}, index=S_val.columns).sort_values("importance", ascending=False)
imp.round(4).head(15)


,importance,direction
repeated_bigram_frac,0.0343,AI lower
hapax_ratio,0.0245,AI higher
n_words,0.0234,AI lower
mattr50,0.0158,AI higher
p_colon,0.0114,AI higher
p_exclam,0.0062,AI higher
punct_per_sent,0.0058,AI higher
zipf_mean,0.0055,AI lower
fw_and,0.0052,AI lower
sent_len_std,0.0045,AI lower


## NB-SVM Component Models


In [9]:
class NBLogReg(ClassifierMixin, BaseEstimator):
    def __init__(self, C=1.0, alpha=1.0):
        # C: regularization strength (smaller = stronger penalty on large weights)
        self.C = C
        self.alpha = alpha

    def fit(self, X, y):
        y = np.asarray(y)
        p = self.alpha + np.asarray(X[y == 1].sum(axis=0)).ravel()
        q = self.alpha + np.asarray(X[y == 0].sum(axis=0)).ravel()

        self.r_ = np.log((p / p.sum()) / (q / q.sum()))
        self.clf_ = LogisticRegression(C=self.C, max_iter=5000, solver="liblinear")
        self.clf_.fit(X.multiply(self.r_).tocsr(), y)
        self.classes_ = self.clf_.classes_
        return self

    def predict_proba(self, X):
        return self.clf_.predict_proba(X.multiply(self.r_).tocsr())

    def predict(self, X):
        return self.clf_.predict(X.multiply(self.r_).tocsr())

In [10]:
C_GRID = [0.01, 0.03, 0.1, 0.3, 1.0]

def run_specialist(name, make_vectorizer):
    t0 = time.time()
    # Learn the vocabulary from the 8,000 training rows only, then apply it to validation
    vec = make_vectorizer()
    A_tr = vec.fit_transform(X_tr)
    A_val = vec.transform(X_val)
    print(f"{name}: {A_tr.shape[1]:,} features")

    # Pick C by 5-fold CV on the training rows; the best C is refit on all 8,000
    gs = GridSearchCV(NBLogReg(), {"C": C_GRID}, scoring="roc_auc", cv=cv, n_jobs=-1)
    gs.fit(A_tr, y_tr)
    model = gs.best_estimator_
    record(name, gs.best_score_, model.predict_proba(A_val)[:, 1], gs.best_params_, time.time() - t0)
    return vec, model


## Models 1–4: four NB-SVM specialists
Each base component model reads a different part of the text:

| Component Model | Reads | Example |
|---|---|---|
| Word | words and word pairs | "delve into" |
| Char | 3–5-letter pieces inside words | "-ization", typos |
| Skeleton | function words + punctuation only; other words → x / Xc / 0 | ", and it is x to x" |
| Cross-boundary char | 1–5-character pieces across spaces and punctuation | ", which", " -- " |


In [11]:
def word_vectorizer():
    # Words and word pairs that appear in at least 2 passages and in at most 95% of them
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95,
                           binary=True, use_idf=False, norm=None)

word_vec, word_nb = run_specialist("nbsvm_word", word_vectorizer)


nbsvm_word: 167,947 features
nbsvm_word: CV AUROC 0.9436, val 0.9512, {'C': 0.3} | 4s


In [12]:
def char_vectorizer():
    # char_wb = 3-5 character pieces taken inside word boundaries; keep the 300k most common
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_features=300_000,
                           binary=True, use_idf=False, norm=None)

char_vec, char_nb = run_specialist("nbsvm_char", char_vectorizer)


nbsvm_char: 132,980 features
nbsvm_char: CV AUROC 0.9440, val 0.9535, {'C': 0.03} | 24s


In [13]:
# Tokens: words (with apostrophes), numbers, or single punctuation marks
TOK_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?|\d+|[^\sA-Za-z\d]")

def skeleton(text):
    out = []
    for tok in TOK_RE.findall(text):
        low = tok.lower()
        if low in FUNCTION_WORDS:
            out.append(low)
        elif tok[0].isdigit():
            out.append("0")
        elif tok[0].isalpha():
            # Content word: keep only whether it was capitalized
            out.append("Xc" if tok[0].isupper() else "x")
        else:
            out.append(tok)
    return " ".join(out)

print(skeleton("The stew offers a unique blend of flavors, and it is easy to make."))

def skeleton_vectorizer():
    # Read the skeleton as space-separated tokens, 1-4 at a time; keep case so x and Xc stay different
    return TfidfVectorizer(preprocessor=skeleton, token_pattern=r"\S+", lowercase=False,
                           ngram_range=(1, 4), min_df=3, binary=True, use_idf=False, norm=None)

skel_vec, skel_nb = run_specialist("nbsvm_skeleton", skeleton_vectorizer)

# Effective weight of each pattern = NB ratio × regression coefficient
eff = skel_nb.clf_.coef_.ravel() * skel_nb.r_
names = skel_vec.get_feature_names_out()
order = np.argsort(eff)
print("Toward AI:   ", " | ".join(names[order[-12:]][::-1]))
print("Toward human:", " | ".join(names[order[:12]]))


the x x a x x of x , and it is x to x .
nbsvm_skeleton: 90,953 features


/Users/abishekchozhan/Documents/school/cs1/aml/kaggle_comp/.venv/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


nbsvm_skeleton: CV AUROC 0.9564, val 0.9668, {'C': 0.1} | 9s
Toward AI:    … | / x / 0 | # # | from x . x | . > | in x to his | ! | : - | # Xc | not x what to | x x - 0 | 0 . in a
Toward human: ( ; | > < x / | Xc ( ; | the x 0 / | . < x / | that in x | a x x i | x Xc the x | , 0 ) , | $ - x . | ) 0 x Xc | < x / >


In [14]:
def charx_vectorizer():
    # analyzer="char" lets pieces run across spaces and punctuation; whitespace collapsed first
    # so blank lines can't be used; case kept
    return TfidfVectorizer(analyzer="char", preprocessor=collapse_text, lowercase=False, ngram_range=(1, 5),
                           min_df=3, max_features=300_000, binary=True, use_idf=False, norm=None)

charx_vec, charx_nb = run_specialist("nbsvm_charx", charx_vectorizer)


nbsvm_charx: 300,000 features
nbsvm_charx: CV AUROC 0.9663, val 0.9716, {'C': 0.1} | 66s


## Combining the five models: rank-averaging
Each model's scores are turned into ranks between 0 and 1 (the lowest score gets about 0, the highest
gets 1), and the five rank columns are averaged. The stylometric model gets an extra weight, because it is
the only model that reads style numbers rather than n-grams.

The weight is chosen using out-of-fold scores on the 8,000 training rows. Each training passage
gets scored by a model that did not train on it, so we can test weights without touching validation.

`RANK_AVG` in the first cell switches between this rank-average (`True`) and a plain average of the
models' probabilities (`False`). The submission and report are written on the plain average of the probabilities as it was simpler and more stable.


In [15]:
from sklearn.model_selection import cross_val_predict
from scipy.stats import rankdata

t0 = time.time()
SPECIALISTS = {"nbsvm_word": (word_vec, word_nb), "nbsvm_char": (char_vec, char_nb),
               "nbsvm_skeleton": (skel_vec, skel_nb), "nbsvm_charx": (charx_vec, charx_nb)}

oof_scores = {}
for name, (vec, model) in SPECIALISTS.items():
    A_tr = vec.transform(X_tr)
    oof_scores[name] = cross_val_predict(NBLogReg(C=model.C), A_tr, y_tr, cv=cv,
                                         method="predict_proba", n_jobs=-1)[:, 1]
oof_scores["stylo_hgb"] = cross_val_predict(
    HistGradientBoostingClassifier(max_iter=400, random_state=SEED, **stylo_hgb_params),
    S_tr, y_tr, cv=cv, method="predict_proba")[:, 1]

for name, s in oof_scores.items():
    print(f"{name:15s} out-of-fold AUROC {roc_auc_score(y_tr, s):.4f}")

def rank01(x):
    return rankdata(x) / len(x)

def rank_avg(scores, names, style_w):
    weights = [style_w if n == "stylo_hgb" else 1.0 for n in names]
    return np.average([rank01(scores[n]) for n in names], axis=0, weights=weights)

def prob_avg(scores, names, style_w):
    weights = [style_w if n == "stylo_hgb" else 1.0 for n in names]
    return np.average([scores[n] for n in names], axis=0, weights=weights)

def combine(scores, names, style_w):
    if RANK_AVG:
        return rank_avg(scores, names, style_w)
    return prob_avg(scores, names, style_w)

COMBINE_NAME = "rank_average" if RANK_AVG else "prob_average"
MODELS = ["nbsvm_word", "nbsvm_char", "nbsvm_skeleton", "nbsvm_charx", "stylo_hgb"]
STYLE_W_GRID = [1, 2, 3, 4, 5]

oof_auc = {w: roc_auc_score(y_tr, combine(oof_scores, MODELS, w)) for w in STYLE_W_GRID}
for w, a in oof_auc.items():
    print(f"style weight {w}: out-of-fold AUROC {a:.4f}")
STYLE_W = max(oof_auc, key=oof_auc.get)

record(COMBINE_NAME, oof_auc[STYLE_W], combine(val_scores, MODELS, STYLE_W),
       {"style_w": STYLE_W}, time.time() - t0)


nbsvm_word      out-of-fold AUROC 0.9435
nbsvm_char      out-of-fold AUROC 0.9441
nbsvm_skeleton  out-of-fold AUROC 0.9564
nbsvm_charx     out-of-fold AUROC 0.9664
stylo_hgb       out-of-fold AUROC 0.9605
style weight 1: out-of-fold AUROC 0.9805
style weight 2: out-of-fold AUROC 0.9820
style weight 3: out-of-fold AUROC 0.9813
style weight 4: out-of-fold AUROC 0.9800
style weight 5: out-of-fold AUROC 0.9789
prob_average: CV AUROC 0.9820, val 0.9888, {'style_w': 2} | 57s


In [16]:
display(pd.DataFrame(results).drop(columns="seconds"))

full_auc = roc_auc_score(y_val, combine(val_scores, MODELS, STYLE_W))
rows = []
for m in MODELS:
    rest = [n for n in MODELS if n != m]
    rows.append({"left out": m, "val_auc": roc_auc_score(y_val, combine(val_scores, rest, STYLE_W))})
ablation = pd.DataFrame(rows)
ablation["change"] = ablation["val_auc"] - full_auc
ablation.round(4)


,model,cv_auc,val_auc,params
0,stylo_hgb,0.9606,0.9701,"{'learning_rate': 0.1, 'max_leaf_nodes': 31}"
1,nbsvm_word,0.9436,0.9512,{'C': 0.3}
2,nbsvm_char,0.9440,0.9535,{'C': 0.03}
3,nbsvm_skeleton,0.9564,0.9668,{'C': 0.1}
4,nbsvm_charx,0.9663,0.9716,{'C': 0.1}
5,prob_average,0.9820,0.9888,{'style_w': 2}


,left out,val_auc,change
0,nbsvm_word,0.9881,-0.0006
1,nbsvm_char,0.9887,-0.0001
2,nbsvm_skeleton,0.9870,-0.0018
3,nbsvm_charx,0.9884,-0.0004
4,stylo_hgb,0.9794,-0.0094


## Final model
For the final predictions, every model is refit on all 10,000 labeled rows with the settings chosen above (each specialist's C, the HGB settings and the
style weight).

test.csv is only ever predicted. No part of the model is ever fit on that data.


In [17]:
t0 = time.time()
VECTORIZERS = {"nbsvm_word": word_vectorizer, "nbsvm_char": char_vectorizer,
               "nbsvm_skeleton": skeleton_vectorizer, "nbsvm_charx": charx_vectorizer}

test_scores = {}
for name, make_vectorizer in VECTORIZERS.items():
    vec = make_vectorizer()
    A_all = vec.fit_transform(X_all)
    A_test = vec.transform(X_test)
    best_C = SPECIALISTS[name][1].C
    model = NBLogReg(C=best_C).fit(A_all, y_all)
    test_scores[name] = model.predict_proba(A_test)[:, 1]
    print(f"{name:15s} refit on {len(X_all):,} rows (C={best_C}) | {time.time() - t0:.0f}s")

S_all = stylo_frame(X_all)
S_test = stylo_frame(X_test)
stylo_final = HistGradientBoostingClassifier(max_iter=400, random_state=SEED, **stylo_hgb_params)
stylo_final.fit(S_all, y_all)
test_scores["stylo_hgb"] = stylo_final.predict_proba(S_test)[:, 1]
print(f"stylo_hgb       refit on {len(X_all):,} rows | {time.time() - t0:.0f}s")

# Same combining method and style weight as on validation
test_pred = combine(test_scores, MODELS, STYLE_W)


nbsvm_word      refit on 10,000 rows (C=0.3) | 3s
nbsvm_char      refit on 10,000 rows (C=0.03) | 16s
nbsvm_skeleton  refit on 10,000 rows (C=0.1) | 23s
nbsvm_charx     refit on 10,000 rows (C=0.1) | 54s
stylo_hgb       refit on 10,000 rows | 70s


In [18]:
sub = pd.DataFrame({"id": test_df["id"], "label": test_pred})
out_path = SUB_DIR / "submission.csv"
sub.to_csv(out_path, index=False)

sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
assert list(sub.columns) == list(sample.columns)
assert len(sub) == len(sample) and (sub["id"].values == sample["id"].values).all()
assert sub["label"].between(0, 1).all() and sub["label"].notna().all()

print(f"wrote {out_path}: {len(sub):,} rows | mean score {sub['label'].mean():.3f}")
sub.head()


wrote submissions/submission.csv: 3,000 rows | mean score 0.445


,id,label
0,13001,0.993365
1,13002,0.190369
2,13003,0.312839
3,13004,0.380391
4,13005,0.985326


## Error analysis: 5 false positives and 5 false negatives

In [19]:
import textwrap

REVIEW_IDS = [3326, 7562, 6187, 3660, 5227, 9490, 1548, 3771, 3062, 4600]
lookup = train_df.set_index("id")
for i in REVIEW_IDS:
    text = lookup.loc[i, "text"]
    label = "AI" if lookup.loc[i, "label"] == 1 else "human"
    print("=" * 90)
    print(f"id {i} | true label: {label} | {len(text.split())} words")
    print("=" * 90)
    # Wrap each original line at 90 characters, keeping the passage's own line breaks
    for line in text.split("\n"):
        print(textwrap.fill(line, width=90) if line.strip() else "")
    print()


id 3326 | true label: human | 114 words
The Academy Award winning war epic Braveheart is an extraordinary film that set a new bar
for the genre. The story follows a Scottish peasant named William Wallace who's pushed
into leading a rebellion against the tyranny of English rule when his wife is murdered.
Starring Mel Gibson, Patrick McGoohan, Sophie Marceau, and Brendan Gleeson, the film has a
strong cast that delivers outstanding performances. Additionally, the battle sequences are
remarkably gritty and violent, which adds to the authentic tone of the film. And, composer
James Horner provides a sweeping and majestic score that's incredibly powerful. A
groundbreaking film, Braveheart does an exceptional job at depicting both the valor and
the horror of war.

id 7562 | true label: human | 280 words
1 c. sliced ripe banana
1/3 c. buttermilk
1 1/2 Tbsp. oil
1 1/2 Tbsp. honey
1 1/3 c. Robin Hood Best For Bread homestyle white flour
2/3 c. Robin Hood Best For Bread whole wheat flour
1 tsp sa